In [3]:
# Importing libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Loading dataset

In [6]:
df = pd.read_csv(r"C:\Users\J.Shiva\OneDrive\Attachments\credit-risk-prediction\data\raw\loan_payments.csv")
print("Shape:", df.shape)
df.head()

Shape: (54231, 43)


,id,member_id,loan_amount,funded_amount,funded_amount_inv,term,int_rate,instalment,grade,sub_grade,...,recoveries,collection_recovery_fee,last_payment_date,last_payment_amount,next_payment_date,last_credit_pull_date,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type
0,38676116,41461848,8000,8000.0,8000.0,36 months,7.49,248.82,A,A4,...,0.0,0.0,Jan-2022,248.82,Feb-2022,Jan-2022,0.0,5.0,1,INDIVIDUAL
1,38656203,41440010,13200,13200.0,13200.0,36 months,6.99,407.52,A,A3,...,0.0,0.0,Jan-2022,407.52,Feb-2022,Jan-2022,0.0,NaN,1,INDIVIDUAL
2,38656154,41439961,16000,16000.0,16000.0,36 months,7.49,497.63,A,A4,...,0.0,0.0,Oct-2021,12850.16,NaN,Oct-2021,0.0,NaN,1,INDIVIDUAL
3,38656128,41439934,15000,15000.0,15000.0,36 months,14.31,514.93,C,C4,...,0.0,0.0,Jun-2021,13899.67,NaN,Jun-2021,0.0,NaN,1,INDIVIDUAL
4,38656121,41439927,15000,15000.0,15000.0,36 months,6.03,456.54,A,A1,...,0.0,0.0,Jan-2022,456.54,Feb-2022,Jan-2022,0.0,NaN,1,INDIVIDUAL


In [7]:
# Creating binary target column

In [8]:
default_statuses = [
    'Charged Off',
    'Does not meet the credit policy. Status:Charged Off',
    'Late (31-120 days)',
    'Late (16-30 days)',
    'Default',
    'In Grace Period'
]

df['target'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)
print("Target created:", df['target'].value_counts().to_dict())

Target created: {0: 47289, 1: 6942}


In [9]:
# Dropping useless columns

In [10]:
drop_cols = [
    'id', 'member_id', 'policy_code', 'loan_status', 'payment_plan',
    'mths_since_last_record', 'mths_since_last_major_derog',
    'next_payment_date', 'mths_since_last_delinq',
    'issue_date', 'earliest_credit_line', 'last_payment_date', 'last_credit_pull_date',
    'funded_amount', 'funded_amount_inv', 'out_prncp_inv', 'total_payment_inv',
    'recoveries', 'collection_recovery_fee', 'total_rec_late_fee'
]

df.drop(columns=drop_cols, inplace=True)
print("Shape after dropping:", df.shape)
print("Remaining columns:", df.columns.tolist())

Shape after dropping: (54231, 24)
Remaining columns: ['loan_amount', 'term', 'int_rate', 'instalment', 'grade', 'sub_grade', 'employment_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_accounts', 'total_accounts', 'out_prncp', 'total_payment', 'total_rec_prncp', 'total_rec_int', 'last_payment_amount', 'collections_12_mths_ex_med', 'application_type', 'target']


In [11]:
# Checking remaining null values

In [12]:
null_df = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(null_df[null_df['Null Count'] > 0].sort_values('Null %', ascending=False))

                            Null Count  Null %
int_rate                          5169    9.53
term                              4772    8.80
employment_length                 2118    3.91
collections_12_mths_ex_med          51    0.09


In [13]:
# Fixing arrow string nulls using pandas NA check

In [14]:
term_mode = df['term'].dropna().mode()[0]
emp_mode = df['employment_length'].dropna().mode()[0]

df['term'] = df['term'].apply(lambda x: term_mode if pd.isna(x) else x)
df['employment_length'] = df['employment_length'].apply(lambda x: emp_mode if pd.isna(x) else x)

print("Nulls remaining:", df.isnull().sum().sum())
print("\nNull breakdown:\n", df.isnull().sum()[df.isnull().sum() > 0])

Nulls remaining: 5220

Null breakdown:
 int_rate                      5169
collections_12_mths_ex_med      51
dtype: int64


In [15]:
# Fixing null imputation for int_rate and collections_12_mths_ex_med

In [16]:
int_rate_median = df['int_rate'].dropna().median()
col_median = df['collections_12_mths_ex_med'].dropna().median()

df['int_rate'] = df['int_rate'].apply(lambda x: int_rate_median if pd.isna(x) else x)
df['collections_12_mths_ex_med'] = df['collections_12_mths_ex_med'].apply(lambda x: col_median if pd.isna(x) else x)

print("Nulls remaining:", df.isnull().sum().sum())

Nulls remaining: 0


In [17]:
# Checking categorical columns unique values

In [18]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)
for col in cat_cols:
    print(f"\n{col} — {df[col].nunique()} unique values:", df[col].unique().tolist())

Categorical columns: ['term', 'grade', 'sub_grade', 'employment_length', 'home_ownership', 'verification_status', 'purpose', 'application_type']

term — 2 unique values: ['36 months', '60 months']

grade — 7 unique values: ['A', 'C', 'B', 'E', 'F', 'D', 'G']

sub_grade — 35 unique values: ['A4', 'A3', 'C4', 'A1', 'B4', 'E5', 'E3', 'C2', 'A5', 'B3', 'C1', 'C3', 'F1', 'D2', 'F4', 'B2', 'D1', 'E2', 'A2', 'D3', 'D4', 'E1', 'B1', 'D5', 'C5', 'B5', 'G1', 'G3', 'F2', 'E4', 'F5', 'G2', 'F3', 'G4', 'G5']

employment_length — 11 unique values: ['5 years', '9 years', '8 years', '1 year', '10+ years', '< 1 year', '7 years', '3 years', '4 years', '6 years', '2 years']

home_ownership — 5 unique values: ['MORTGAGE', 'RENT', 'OWN', 'OTHER', 'NONE']

verification_status — 3 unique values: ['Not Verified', 'Source Verified', 'Verified']

purpose — 14 unique values: ['credit_card', 'debt_consolidation', 'home_improvement', 'small_business', 'renewable_energy', 'major_purchase', 'other', 'moving', 'car',

In [19]:
# Dropping zero variance and redundant columns

In [20]:
df.drop(columns=['application_type', 'sub_grade'], inplace=True)
print("Shape after dropping:", df.shape)

Shape after dropping: (54231, 22)


In [21]:
# Encoding term and employment_length columns

In [22]:
df['term'] = df['term'].str.extract('(\d+)').astype(int)

emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
df['employment_length'] = df['employment_length'].map(emp_map)

print("term unique:", df['term'].unique())
print("employment_length unique:", sorted(df['employment_length'].unique()))

term unique: [36 60]
employment_length unique: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [23]:
# Encoding grade column ordinally

In [24]:
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade'] = df['grade'].map(grade_map)

print("grade unique:", sorted(df['grade'].unique()))

grade unique: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [25]:
# Label encoding remaining categorical columns

In [26]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
cat_cols = ['home_ownership', 'verification_status', 'purpose']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Encoding done!")
print(df[cat_cols].head())

Encoding done!
   home_ownership  verification_status  purpose
0               0                    0        1
1               4                    0        1
2               0                    1        1
3               4                    1        2
4               0                    2        2


In [27]:
# Checking dtypes after encoding

In [28]:
print(df.dtypes)
print("\nShape:", df.shape)

loan_amount                     int64
term                            int64
int_rate                      float64
instalment                    float64
grade                           int64
employment_length               int64
home_ownership                  int64
annual_inc                    float64
verification_status             int64
purpose                         int64
dti                           float64
delinq_2yrs                     int64
inq_last_6mths                  int64
open_accounts                   int64
total_accounts                  int64
out_prncp                     float64
total_payment                 float64
total_rec_prncp               float64
total_rec_int                 float64
last_payment_amount           float64
collections_12_mths_ex_med    float64
target                          int64
dtype: object

Shape: (54231, 22)


In [29]:
# Checking skewness of numerical columns

In [30]:
skew_df = df.drop(columns=['target']).skew().sort_values(ascending=False)
print(skew_df)

collections_12_mths_ex_med    20.262376
annual_inc                     8.711831
delinq_2yrs                    5.370002
inq_last_6mths                 3.248918
last_payment_amount            2.499381
out_prncp                      2.356426
total_rec_int                  2.204322
purpose                        1.901034
total_payment                  1.267891
total_rec_prncp                1.261015
term                           1.148358
open_accounts                  1.059282
instalment                     0.996981
loan_amount                    0.805259
total_accounts                 0.779014
grade                          0.695002
int_rate                       0.456515
dti                            0.189420
home_ownership                 0.039228
verification_status           -0.112004
employment_length             -0.183019
dtype: float64


Highly skewed columns clearly visible! Anything above 1.0 needs log transformation:
# Applying log transformation to skewed columns

In [33]:
skewed_cols = ['collections_12_mths_ex_med', 'annual_inc', 'delinq_2yrs',
               'inq_last_6mths', 'last_payment_amount', 'out_prncp',
               'total_rec_int', 'total_payment', 'total_rec_prncp']

for col in skewed_cols:
    df[col] = np.log1p(df[col])

print("Skewness after transformation:")
print(df[skewed_cols].skew().sort_values(ascending=False))

Skewness after transformation:
collections_12_mths_ex_med    16.451622
delinq_2yrs                    2.173166
out_prncp                      0.531961
inq_last_6mths                 0.351262
annual_inc                    -0.071007
last_payment_amount           -1.581159
total_rec_int                 -4.322484
total_payment                 -6.651500
total_rec_prncp               -8.669144
dtype: float64


In [34]:
# Applying sqrt transformation to still-skewed columns

In [35]:
still_skewed = ['collections_12_mths_ex_med', 'delinq_2yrs']

for col in still_skewed:
    df[col] = np.sqrt(df[col])

print("Skewness after sqrt transformation:")
print(df[still_skewed].skew().sort_values(ascending=False))

Skewness after sqrt transformation:
collections_12_mths_ex_med    16.096579
delinq_2yrs                    1.933759
dtype: float64


In [36]:
# Checking final skewness of all columns

In [37]:
final_skew = df.drop(columns=['target']).skew().sort_values(ascending=False)
print(final_skew)

collections_12_mths_ex_med    16.096579
delinq_2yrs                    1.933759
purpose                        1.901034
term                           1.148358
open_accounts                  1.059282
instalment                     0.996981
loan_amount                    0.805259
total_accounts                 0.779014
grade                          0.695002
out_prncp                      0.531961
int_rate                       0.456515
inq_last_6mths                 0.351262
dti                            0.189420
home_ownership                 0.039228
annual_inc                    -0.071007
verification_status           -0.112004
employment_length             -0.183019
last_payment_amount           -1.581159
total_rec_int                 -4.322484
total_payment                 -6.651500
total_rec_prncp               -8.669144
dtype: float64


In [38]:
# Saving preprocessed dataset

In [39]:
save_path = r"C:\Users\J.Shiva\OneDrive\Attachments\credit-risk-prediction\data\processed\loan_payments_preprocessed.csv"
df.to_csv(save_path, index=False)
print("Saved successfully!")
print("Shape:", df.shape)

Saved successfully!
Shape: (54231, 22)


In [ ]:
## Notebook 2 Summary — Preprocessing Complete
- Started with 43 columns → reduced to 22 columns
- Created binary target column (0 = Non-Default, 1 = Default)
- Dropped high null columns (>50%), identifiers, date columns, leakage columns
- Dropped zero variance column (application_type) and redundant column (sub_grade)
- Imputed nulls — median for numerical, mode for categorical
- Encoded term and employment_length ordinallym
- Encoded grade ordinally (A=1 to G=7)
- Label encoded home_ownership, verification_status, purpose
- Applied log1p transformation to highly skewed columns
- Saved preprocessed data to data/processed/ folder